# build User Data

In [ ]:
import json
# 建構使用者
# 使用者內容包括
class User:
    def __init__(self, username, password, email):
        self.username = username
        self.password = password
        self.email = email
# 輸入：num_users：數量, startNumber：從幾號開始取名，取名的方式
def create_users_json(num_users,startNumber):
    users = []
    startNumber-=1
    for i in range(1, num_users+1):
        user = User(str(i+startNumber), str(i+startNumber), f"redacted-user@example.com")
        users.append(user.__dict__)
    return json.dumps(users)

In [ ]:
# 建構 3 個使用者的 JSON 格式變數
json_var = create_users_json(10,1)
print(json_var)

# insert_many data to mongodb

In [ ]:
import pymongo

def insert_data_to_mongodb(data, uri='localhost:27017', database='sample_users', collection='accounts'):
    client = pymongo.MongoClient(uri)
    db = client[database]
    col = db[collection]
    try:
        col.insert_many(data)
        print("Data insertion successful!")
        return True
    except Exception as e:
        print(f"Data insertion failed: {e}")
        return False
    finally:
        client.close()

In [ ]:
# 因為 create_users_json() 返回的是以下 JSON 字串
data_str = create_users_json(1000,1)
# 將 JSON 字串解析為 Python 對象
data = json.loads(data_str)
# insert_many to mongodb
insert_data_to_mongodb(data)

# mongodb opration

In [ ]:
import pymongo
from bson.objectid import ObjectId

class MongoDB:
    # 自記去指定，因為易對錯誤的資料庫位置做操作
    def __init__(self, uri='localhost:27017', database='sample_users', collection='browsing_histories'):
        self.uri = uri
        self.database = database
        self.collection = collection
        self.client = pymongo.MongoClient(self.uri)
        self.db = self.client[self.database]
        self.col = self.db[self.collection]

    def insert_data(self, data):
        try:
            result = self.col.insert_many(data)
            print("Data insertion successful!")
            return result.inserted_ids
        except Exception as e:
            print(f"Data insertion failed: {e}")
            return None
    def insert_data_one(self, data):
        try:
            result = self.col.insert_one(data)
#             print("Data insertion successful!")
            return result.inserted_id
        except Exception as e:
            print(f"Data insertion failed: {e}")
            return None
    def find_data(self, query={}, projection=None, limit=0):
        cursor = self.col.find(query, projection=projection, limit=limit)
#         return cursor
        return [doc for doc in cursor]
    def update_data(self, id, update_data):
        try:
            self.col.update_one({'_id': ObjectId(id)}, {'$set': update_data})
            print('Data updated successfully')
        except Exception as e:
            print(f'Data update failed: {e}')
# # Example Usage - update_data
# mongo = MongoDB(uri='localhost:27017', database='sample_users', collection='accounts')
# # Suppose we want to update the document with _id '60b1e9ac0d0fa34d2d2e51c4'
# update_data = {'name': 'John', 'age': 30, 'email': 'john@example.com'}
# mongo.update_data('60b1e9ac0d0fa34d2d2e51c4', update_data)
# mongo.close()
    def close(self):
        self.client.close()
    def get_max_EventID0308(self):
        query = {"EventID0308": {"$exists": True}}
        projection = {"EventID0308": 1, "_id": 0}
        cursor = self.col.find(query, projection).sort("EventID0308", pymongo.DESCENDING).limit(1)
        result = list(cursor)
        if result:
            max_event_id = result[0]["EventID0308"]
            return max_event_id
        else:
            print("No EventID0308 found in collection")
            return Non


In [ ]:
    def take_idAndEventID0308FromOrinArticle():
        mongo = MongoDB('localhost:27017', 'sample_event_DB', collection='sample_event_Coll_copy')
        data = mongo.find_data({},{'_id':1,'EventID0308':1},3)
        # max_EventID0308=mongo.get_max_EventID0308()
        mongo.close()

In [ ]:
mongo = MongoDB()
users_id = mongo.find_data({},{'_id':1})
print(users_id)
mongo.close()

In [ ]:
mongo = MongoDB('localhost:27017', 'sample_event_DB', collection='sample_event_Coll_copy')
data = mongo.find_data({'EventID0308':1},{'_id':1,'EventID0308':1})
# max_EventID0308=mongo.get_max_EventID0308()
mongo.close()
print(data)

# build the User Browsing History

In [ ]:
import random
import pymongo
from datetime import datetime
from bson.objectid import ObjectId

class JSONGenerator_UserBrowsingHistory:
    def __init__(self, uri='localhost:27017', database='topic_collection', collection='original'):
        self.uri = uri
        self.database = database
        self.collection = collection
        self.client = pymongo.MongoClient(self.uri)
        self.db = self.client[self.database]
        self.col = self.db[self.collection]
    
    def generate_json(self):
        data = {}
        # get random document from collection
        cursor = self.col.find({}, {'_id': 1})
        doc = random.choice([d for d in cursor])
        data['original_id'] = doc['_id']# bson format
#         print(f'doc _id from db type is: {type(doc["_id"])}\n')
#         data['EventID'] = str(doc['EventID0308'])
        # generate random view time
        view_time = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S.%fZ')
        data['view_time'] = view_time #'2022-04-01T12:00:00.000Z'
        # generate random browsing duration
        data['browsing_duration'] = random.randint(0, 30)
        # generate random topic
        data['topic'] = random.randint(0, 3)
        return data
    
    def find_data(self, query={}, projection=None, limit=0):
        cursor = self.col.find(query, projection=projection, limit=limit)
        return [doc for doc in cursor]
    
    def close(self):
        self.client.close()

In [ ]:
# importing ObjectId from bson library
from bson.objectid import ObjectId
json_generator = JSONGenerator_UserBrowsingHistory()
json_data = json_generator.generate_json()
print(f'make a json_data: {json_data}\n')
print(f'json_data type is: {type(json_data)}\n')
print(f'take the original_id: {json_data["original_id"]}\n')
print(f'original_id type is: {type(json_data["original_id"])}\n')
# print(f'take the EventID: {json_data["EventID"]}\n')
# print(f'EventID type is: {type(json_data["EventID"])}\n')
query = {"_id": json_data["original_id"]} # must be int() when you find with "EventID" !!!
result = json_generator.find_data(query,{'_id':1})
print('result with _id only: \n',result)
print(f'only _id type is: {type(json_data["original_id"])}')# It is <class 'bson.objectid.ObjectId'> !!! special format

query = {"_id":  ObjectId(json_data["original_id"])}
result = json_generator.find_data(query,{'_id':1})  # must be ObjectId() when you find with "EventID" !!!
print('result with ObjectId(_id): \n',result)
print(f'ObjectId type is: {type(ObjectId(json_data["original_id"]))}')# It is <class 'bson.objectid.ObjectId'> !!! special format

query = {"_id":  json_data["original_id"]}#original_id()有無都沒差
result = json_generator.find_data(query) 
print('all result: \n',result)

In [ ]:
from bson.objectid import ObjectId
import random
# take all user _id from mongofb
mongo = MongoDB(uri='localhost:27017', database='sample_users', collection='accounts')
Allusers_id = mongo.find_data({},{'_id':1})
mongo.close()
# destination of storing
mongo = MongoDB(uri='localhost:27017', database='sample_users', collection='browsing_histories')

# print(type(Allusers_id))#<class 'dict'>
# print(len(Allusers_id))# 1000

# counterFor=0
for user_id in Allusers_id:
#     print(type(users_id))#<class 'dict'>
#     if counterFor==3:
#         break
#     counterFor+=1
    # take user _id
    userBrowsingHistory=user_id
    
#     if mongo.find_data({'_id': user_id['_id']}):
#         print(f"Data with _id {user_id['_id']} already exists.")
#         continue
    # make Browsing History
    json_generator = JSONGenerator_UserBrowsingHistory()
    All_json_data = []
    for xthBrowsingHistory in range(0, random.randint(1, 30)):# []
        json_data = json_generator.generate_json()
        All_json_data.append(json_data)
    userBrowsingHistory.update({"browsing_history": All_json_data}) 
    mongo.insert_data_one(userBrowsingHistory)
#     print('\n\n',userBrowsingHistory)
mongo.close()

## 覆寫資料(最後才能測試，知道相似度推薦是否準確)

In [ ]:
from bson.objectid import ObjectId
from datetime import datetime

mongo = MongoDB(uri='localhost:27017', database='sample_users', collection='browsing_histories')
# Suppose we want to update the document with _id '60b1e9ac0d0fa34d2d2e51c4'
update_data = {'browsing_history': [
    {
        'original_id': ObjectId("64242d936ccbd541cce53809"),
        'view_time': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S.%fZ'),
        'browsing_duration': 10,
        'topic': 1
      },
      {
        'original_id': ObjectId("64242d8c6ccbd541cce51e63"),
        'view_time': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S.%fZ'),
        'browsing_duration': 20,
        'topic': 2
      },
      {
        'original_id': ObjectId("64242d886ccbd541cce5135d"),
        'view_time': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S.%fZ'),
        'browsing_duration': 16,
        'topic': 0
      }
]}
mongo.update_data('64204749c4f28361cf539133', update_data)
mongo.close()

mongo = MongoDB(uri='localhost:27017', database='sample_users', collection='browsing_histories')
# Suppose we want to update the document with _id '60b1e9ac0d0fa34d2d2e51c4'
update_data = {'browsing_history': [
    {
        'original_id': ObjectId("64242d936ccbd541cce53809"),
        'view_time': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S.%fZ'),
        'browsing_duration': 10,
        'topic': 1
      },
      {
        'original_id': ObjectId("64242d8c6ccbd541cce51e63"),
        'view_time': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S.%fZ'),
        'browsing_duration': 20,
        'topic': 2
      }
]}
mongo.update_data('64204749c4f28361cf539134', update_data)
mongo.close()

In [ ]:
from bson.objectid import ObjectId
from datetime import datetime
# 第二集合 覆寫資料
mongo = MongoDB(uri='localhost:27017', database='sample_users', collection='browsing_histories')
# Suppose we want to update the document with _id '60b1e9ac0d0fa34d2d2e51c4'
update_data = {'browsing_history': [
    {
        'original_id': ObjectId("64242d936ccbd541cce53657"),
        'view_time': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S.%fZ'),
        'browsing_duration': 13,
        'topic': 3
      },
      {
        'original_id': ObjectId("64242d8a6ccbd541cce51823"),
        'view_time': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S.%fZ'),
        'browsing_duration': 10,
        'topic': 0
      },
      {
        'original_id': ObjectId("64242d886ccbd541cce513ab"),
        'view_time': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S.%fZ'),
        'browsing_duration': 27,
        'topic': 1
      },
      {
        'original_id': ObjectId("64242d876ccbd541cce50e59"),
        'view_time': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S.%fZ'),
        'browsing_duration': 14,
        'topic': 2
      },
      {
        'original_id': ObjectId("64242d876ccbd541cce50ed9"),
        'view_time': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S.%fZ'),
        'browsing_duration': 21,
        'topic': 2
      }
]}
mongo.update_data('64204749c4f28361cf539135', update_data)
mongo.close()
mongo = MongoDB(uri='localhost:27017', database='sample_users', collection='browsing_histories')
# Suppose we want to update the document with _id '60b1e9ac0d0fa34d2d2e51c4'
update_data = {'browsing_history': [
    {
        'original_id': ObjectId("64242d936ccbd541cce53657"),
        'view_time': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S.%fZ'),
        'browsing_duration': 6,
        'topic': 3
      },
      {
        'original_id': ObjectId("64242d8a6ccbd541cce51823"),
        'view_time': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S.%fZ'),
        'browsing_duration': 19,
        'topic': 0
      },
      {
        'original_id': ObjectId("64242d886ccbd541cce513ab"),
        'view_time': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S.%fZ'),
        'browsing_duration': 3,
        'topic': 1
      },
      {
        'original_id': ObjectId("64242d876ccbd541cce50e59"),
        'view_time': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S.%fZ'),
        'browsing_duration': 14,
        'topic': 2
      }
]}
mongo.update_data('64204749c4f28361cf539136', update_data)
mongo.close()


In [ ]:
from bson.objectid import ObjectId
from datetime import datetime
# 第三集合 覆寫資料
mongo = MongoDB(uri='localhost:27017', database='sample_users', collection='browsing_histories')
# Suppose we want to update the document with _id '60b1e9ac0d0fa34d2d2e51c4'
update_data = {'browsing_history': [
    {
        'original_id': ObjectId("64242d926ccbd541cce53135"),
        'view_time': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S.%fZ'),
        'browsing_duration': 8,
        'topic': 2
      },
      {
        'original_id': ObjectId("64242d8b6ccbd541cce51add"),
        'view_time': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S.%fZ'),
        'browsing_duration': 25,
        'topic': 1
      },
      {
        'original_id': ObjectId("64242d906ccbd541cce52ac7"),
        'view_time': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S.%fZ'),
        'browsing_duration': 22,
        'topic': 0
      },
      {
        'original_id': ObjectId("64242d926ccbd541cce53419"),
        'view_time': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S.%fZ'),
        'browsing_duration': 22,
        'topic': 1
      }
]}
mongo.update_data('64204749c4f28361cf539137', update_data)
mongo.close()
mongo = MongoDB(uri='localhost:27017', database='sample_users', collection='browsing_histories')
# Suppose we want to update the document with _id '60b1e9ac0d0fa34d2d2e51c4'
update_data = {'browsing_history': [# 打亂順序
    {
        'original_id': ObjectId("64242d8b6ccbd541cce51add"),
        'view_time': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S.%fZ'),
        'browsing_duration': 1,
        'topic': 1
    },
    {
        'original_id': ObjectId("64242d926ccbd541cce53419"),
        'view_time': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S.%fZ'),
        'browsing_duration': 29,
        'topic': 1
    },
#     {#要推薦
#         'original_id': ObjectId("64242d906ccbd541cce52ac7"),
#         'view_time': '2023-04-05T04:16:24.866854Z',
#         'browsing_duration': 22,
#         'topic': 0
#     },
    {
        'original_id': ObjectId("64242d926ccbd541cce53135"),
        'view_time':datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S.%fZ'),
        'browsing_duration': 25,
        'topic': 2
    },
]}
mongo.update_data('64204749c4f28361cf53951a', update_data)
mongo.close()


class UserBrowsingHistory:
    def __init__(self, uri='localhost:27017', database='sample_users', collection='browsing_histories'):
        self.uri = uri
        self.database = database
        self.collection = collection
        self.data = None
        
    def load_data(self):
        mongo = MongoDB(uri=self.uri, database=self.database, collection=self.collection)
        data = mongo.find_data(projection={'_id': 1, 'browsing_history.original_id': 1})
        print('data:\n',data)    

ubh = UserBrowsingHistory(uri='localhost:27017', database='sample_users', collection='browsing_histories')
mat = ubh.load_data()
print(mat)

# build matrix(CSC)

In [ ]:
from pymongo import MongoClient
from scipy.sparse import csc_matrix
from bson.objectid import ObjectId

# 建立 MongoDB 連線
client = MongoClient('localhost', 27017)
db = client['sample_users']
collection = db['browsing_histories']

# 查詢資料
data = list(collection.find({}, {'_id': 1, 'browsing_history.original_id': 1}))

# 建立 CSC 格式的稀疏矩陣
user_id_to_index = {}
orig_id_to_index = {}
user_index = 0
orig_index = 0
rows, cols, vals = [], [], []
for d in data:
    _id = d['_id']
    if _id not in user_id_to_index:
        user_id_to_index[_id] = user_index
        user_index += 1
    for bh in d['browsing_history']:
        orig_id = bh['original_id']
        if orig_id not in orig_id_to_index:
            orig_id_to_index[orig_id] = orig_index
            orig_index += 1
        rows.append(user_id_to_index[_id])
        cols.append(orig_id_to_index[orig_id])# col不會重複!
        vals.append(1)


# mat = csc_matrix((vals, (rows, cols))) # row col value is still in COO format
mat = csc_matrix((vals, (rows, cols)), shape=(len(set(rows)), len(set(cols))))
# print(mat)
# 輸出格式是 (row index, column index)\tvalue # 輸出是COO, 方便看

my_list = list(orig_id_to_index.values())
#print(my_list)
def is_in_order(lst):
    """
    檢查 list 中的 int 值是否照順序排列
    """
    for i in range(len(lst)-1):
        if not isinstance(lst[i], int) or not isinstance(lst[i+1], int):
            return False
        if lst[i] > lst[i+1]:
            return False
    return True
print(is_in_order(my_list))  # True

# For CollaborativeFiltering Item-based
from pymongo import MongoClient
from scipy.sparse import csc_matrix
from sklearn.metrics.pairwise import cosine_similarity
from bson.objectid import ObjectId

class MongoCosineSimilarityCalculator:
    def __init__(self, uri, db, collection):
        client = MongoClient(uri)
        self.collection = client[db][collection]
    
    def calculate_cosine_similarity(self, mat):
        # 計算 cosine similarity
        sim_mat = cosine_similarity(mat.T, dense_output=False)
        sim_mat = sim_mat.toarray()

#         # 取得原始 ID 和 user ID
#         orig_id_to_index = {}
#         user_id_to_index = {}
#         for doc in self.collection.find():
#             orig_id_to_index[str(doc['_id'])] = doc['index']
#             for bh in doc['browsing_history']:
#                 user_id_to_index[str(bh['user_id'])] = bh['user_index']

        # 取得相似的原始 ID 和 user ID
        similar_orig_ids = []
        similar_user_ids = []
        for i in range(sim_mat.shape[0]):
#             orig_id = ObjectId(orig_id_to_index[str(i)])
            orig_id = orig_id_to_index[i]
            similar_orig_ids.append({
                'original_id': orig_id,
#                 'similarOriginalList': [ObjectId(orig_id_to_index[str(j)]) for j in sim_mat[i].argsort()[-2:-12:-1] if i != j]
                'similarOriginalList': [orig_id_to_index[j] for j in sim_mat[i].argsort()[-2:-12:-1] if i != j]

            })
#         for i in range(sim_mat.shape[1]):
# #             user_id = ObjectId(user_id_to_index[str(i)])
#             user_id = user_id_to_index[i]
#             similar_user_ids.append({
#                 'user_id': user_id,
#                 'similarUserList': [ObjectId(user_id_to_index[str(j)]) for j in sim_mat[:, i].argsort()[-2:-12:-1] if i != j]
#             })

        return [similar_orig_ids]


# 計算相似度

In [ ]:
from pymongo import MongoClient
from scipy.sparse import csc_matrix
from bson.objectid import ObjectId
import numpy as np
counterFor=0
class CosSimilarity:
    def __init__(self, uri, db, collection):
        self.client = MongoClient(uri)
        self.db = self.client[db]
        self.collection = self.db[collection]
    
    def calculate_similarities(self, mat, destination_uri, destination_db, destination_collection):
        mongoDB = MongoDB(destination_uri, destination_db, destination_collection)
#         result = []

        # 計算每一個 vector 的 norm
        norms = np.sqrt(np.sum(mat.power(2), axis=0))# should axis=0 <- col
        # 輸出是1維矩陣 e.g.
#         第一個向量為[1,2,3,4]，第二個向量為[2,4,6,8]，第三個向量為[3,6,9,12]，則norms的輸出為[3.74165739, 10., 14.86606875]。
        
        # 計算每一個 vector 的相似度
        for i in range(mat.shape[1]):# mat.shape[1] return #cols
            column = mat.getcol(i)
            # 输出 col 的形状和元素`
#             print(column)  # (#rows, 1). suppose (3, 1)
#             print(column.toarray())  # [[0], [2], [0]]
            column_norm = norms[0,i]# take 0th row, ith col value from norms=[3.74165739, 10., 14.86606875]
            
            top_similarities = []
            for j in range(mat.shape[1]):
                if i != j:
                    other_column = mat.getcol(j)

                    other_column_norm = norms[0,j]

                    # 計算 cosine similarity
                    similarity = np.dot(column.T, other_column).toarray()[0][0] / (column_norm * other_column_norm)

                    # 如果相似度為 NaN 則跳過
                    if np.isnan(similarity):
                        continue

                    # 如果還沒有加滿 4 個，或是此相似度比目前最低的還高，就加入 top_similarities
                    if similarity > 0 :
                        # 改用 orig_id_to_index 轉換
                        orig_id = self.get_orig_id_by_index(j)
                        top_similarities.append({'original_id': ObjectId(orig_id), 'similarity': similarity})

            top_similarities.sort(key=lambda x: x['similarity'], reverse=True)
            
            mongoDB.insert_data_one({'original_id': ObjectId(self.get_orig_id_by_index(i)), 'similarOriginalList': top_similarities})
            # 將結果加入 result
#             result.append({'original_id': ObjectId(self.get_orig_id_by_index(i)), 'similarOriginalList': top_similarities})
        mongoDB.close()
#         return result
        return

    # 新增方法
    def get_orig_id_by_index(self, index):
        for orig_id, idx in self.orig_id_to_index.items():
            if idx == index:
                return orig_id
        return None

In [ ]:
print(len(orig_id_to_index))# 1000個使用者只有看到5107筆活動資料

In [ ]:
%%time
uri = 'mongodb://localhost:27017'
db = 'sample_users'
collection = 'browsing_histories'

# 初始化 MongoCosineSimilarityCalculator 物件
calculator = CosSimilarity(uri, db, collection)

# 取得稀疏矩陣
# mat = calculator.get_sparse_matrix()
# set orig_id_to_index
calculator.orig_id_to_index = orig_id_to_index  # 儲存字典

# 計算 cosine similarity(it will store into mongodb autolly)
calculator.calculate_similarities(mat,'mongodb://localhost:27017', 'sample_users', 'similarList_CollaborativeFilteringItem_based3')


# 儲存回mongodb

In [ ]:
uri = 'mongodb://localhost:27017'
db = 'sample_users'
collection = 'similarList_CollaborativeFilteringItem_based'

mongoDB = MongoDB(uri, db, collection)
mongoDB.insert_data(result)
mongoDB.close()

In [ ]:
from scipy.sparse import csc_matrix

# 创建一个稀疏矩阵
data = [1, 2, 3, 4, 5, 6]
indices = [0, 2, 2, 0, 1, 2]
indptr = [0, 2, 3, 6]
mat = csc_matrix((data, indices, indptr), shape=(3, 3))
print(mat)
print("")
# 获取第二列
col = mat.getcol(1)

# 输出 col 的形状和元素
print(col.shape)  # (3, 1)
print(col.toarray())  # [[0], [2], [0]]
